In [1]:
import numpy as np
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, cdist
from scipy.stats import f_oneway

from sklearn.decomposition import PCA

import pandas as pd
import matplotlib.pyplot as plt

import torch

from visuEmbedding import components_to_fig_3D

from typing import Callable, Optional

import plotly.graph_objects as go

# See corelation in cosine similarity

## Some test

In [2]:
def see_corelation_between_df(df1:pd.DataFrame, df2:pd.DataFrame)-> None:
    common_words = df1.index.intersection(df2.index)
    assert len(common_words) > 3, "no suffisent word in common"
    
    common_words = sorted(common_words)
    
    m1_sub = df1.loc[common_words, common_words].values
    m2_sub = df2.loc[common_words, common_words].values
    
    # Get only triangle of 
    upper_indices = np.triu_indices(len(common_words), k=1)
    vec1 = m1_sub[upper_indices]
    vec2 = m2_sub[upper_indices]
    
    corr, p_value = spearmanr(vec1, vec2)
    return corr, p_value

SyntaxError: invalid syntax (2431083103.py, line 11)

# See if same Neighbor

## Jaccard Overlap

In [ ]:
def jaccard_overlap(k:int, words:list[str], vectors1:np.ndarray, vectors2:np.ndarray, batch_size:int = 100):
    num_words = len(words)
    jaccard_scores = []
    
    # Normalize vectors
    vectors1 = vectors1 / (np.linalg.norm(vectors1, axis=1, keepdims=True) + 1e-10)
    vectors2 = vectors2 / (np.linalg.norm(vectors2, axis=1, keepdims=True) + 1e-10)
    

    for i in range(0, num_words, batch_size):
        end = min(i + batch_size, num_words)
        
        batch_1 = vectors1[i:end]
        batch_2 = vectors2[i:end]
        
        sims_1 = np.dot(batch_1, vectors1.T) 
        sims_2 = np.dot(batch_2, vectors2.T)
        
        top_k_idx_1 = np.argsort(sims_1, axis=1)[:, -k-1:-1] # Exclude self (last one)
        top_k_idx_2 = np.argsort(sims_2, axis=1)[:, -k-1:-1]
        
        for j in range(len(batch_1)):
            set_g = set(top_k_idx_1[j])
            set_p = set(top_k_idx_2[j])
            
            intersection = len(set_g.intersection(set_p))
            union = len(set_g.union(set_p))
            
            score = intersection / union
            jaccard_scores.append(score)

    
    return jaccard_scores



In [ ]:
def show_plot_jaccard_score(jaccard_scores:list[float]) -> None:
    average_overlap = np.mean(jaccard_scores)
    
    plt.figure(figsize=(10, 6))

    plt.hist(jaccard_scores, bins=50, color='#1f77b4', alpha=0.7, edgecolor='black', range=(0, 1))

    plt.axvline(average_overlap, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {average_overlap:.3f}')

    plt.title('Distribution of Semantic Overlap (Jaccard Similarity)', fontsize=14)
    plt.xlabel('Jaccard Score (0 = No Overlap, 1 = Perfect Match)', fontsize=12)
    plt.ylabel('Number of Words', fontsize=12)
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.xlim(0, 1)
    plt.show()

## All word

In [ ]:
data_google = np.load('googleW2V/google_sub_embed_all_word.npz')
vecs_google = data_google['vectors']
word_google = data_google['words']

my_model = np.load('one_emb_into_all.npz')
vecs_my_model = my_model['vectors']
word_my_model = my_model['words']

In [ ]:
print(word_google)
print(word_my_model)

In [ ]:
data_google = np.load('googleW2V/google_sub_embed_all_word.npz')
vecs_google = data_google['vectors']

my_model = np.load('one_emb_into_all.npz')
vecs_my_model = my_model['vectors']
word_my_model = my_model['words']

jaccord_score = jaccard_overlap(
    vectors2= vecs_my_model,
    vectors1=vecs_google,
    k=10,
    words=word_my_model # Ou word_google (same)
)

show_plot_jaccard_score(jaccord_score)

In [ ]:
data_google = np.load('googleW2V/google_sub_embed_all_word.npz')
vecs_google = data_google['vectors']

my_model = np.load('original_all.npz')
vecs_my_model = my_model['vectors']

average_overlap = jaccard_overlap(
    vectors2= vecs_my_model,
    vectors1=vecs_google,
    k=10,
    words=word_my_model # Ou word_google (same)
)

show_plot_jaccard_score(average_overlap)

In [ ]:
print(len(vecs_my_model))

## Many plot of Jaccard Neighbor

In [ ]:
import math

def show_multi_plot_jaccard(model_scores: dict[str, list[float]], nb_cols:int=2) -> None:
    """
    Chat GPT function
    Plots a grid of histograms comparing Jaccard scores from multiple models.
    
    Args:
        model_scores: A dictionary where Key = Model Name, Value = List of Jaccard Scores
    """
    num_models = len(model_scores)
    
    cols = nb_cols
    rows = math.ceil(num_models / cols)
    
    # Create the figure
    # sharex=True makes comparison easier (all axes align)
    fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows), sharex=True, sharey=False)
    
    # Flatten axes array for easy iteration (handles 1D or 2D arrays)
    if num_models > 1:
        axes = axes.flatten()
    else:
        axes = [axes] # Handle single model case gracefully

    # 2. Iterate and Plot
    for i, (model_name, scores) in enumerate(model_scores.items()):
        ax = axes[i]
        scores_arr = np.array(scores)
        mean_score = np.mean(scores_arr)
        
        # A. Histogram
        ax.hist(scores_arr, bins=50, color='#1f77b4', alpha=0.7, 
                edgecolor='black', range=(0, 1))
        
        # B. Mean Line
        ax.axvline(mean_score, color='red', linestyle='dashed', linewidth=2, 
                   label=f'Mean: {mean_score:.3f}')
        
        # C. Styling
        ax.set_title(f'{model_name}', fontsize=12, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(axis='y', alpha=0.3)
        ax.set_xlim(0, 1)
        
        # Only add Y label to the first column to reduce clutter
        if i % cols == 0:
            ax.set_ylabel('Count of Words')

    # 3. Clean up empty plots
    # If you have 3 models in a 2x2 grid, the 4th plot should be hidden
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    # Common X label at the bottom
    fig.text(0.5, 0.01, 'Jaccard Score (0 = No Overlap, 1 = Perfect Match)', 
             ha='center', fontsize=12)

    plt.tight_layout()
    # Adjust slightly to make room for the common X label
    plt.subplots_adjust(bottom=0.08) 
    plt.show()

### Comparison avec le W2V de google

In [ ]:
# Important words
google_important_word = np.load('googleW2V/google_sub_embed.npz')
vecs_google_important_word = google_important_word['vectors']
words_gg_iw = google_important_word['words']
print(f"For model : google_important_word , we have : {len(vecs_google_important_word)} words")

one_emb_intonation_important_word = np.load('one_emb_into_imp.npz')
vecs_one_emb_intonation_important_word = one_emb_intonation_important_word['vectors']
words_oei_iw = one_emb_intonation_important_word['words']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_one_emb_intonation_important_word)} words")

original_important_word = np.load('original_imp.npz')
vecs_original_important_word = original_important_word['vectors']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_original_important_word)} words")

one_emb_important_word = np.load('one_emb_imp.npz')
vecs_one_emb_important_word = one_emb_important_word['vectors']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_one_emb_important_word)} words")

# All words
google_all_word = np.load('googleW2V/google_sub_embed_all_word.npz')
vecs_google_all_word = google_all_word['vectors']
words = google_all_word['words']
print(f"For model : google_important_word , we have : {len(vecs_google_all_word)} words")

one_emb_intonation_all_word = np.load('one_emb_into_all.npz')
vecs_one_emb_intonation_all_word = one_emb_intonation_all_word['vectors']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_one_emb_intonation_all_word)} words")

original_all_word = np.load('original_all.npz')
vecs_original_all_word = original_all_word['vectors']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_original_all_word)} words")

one_emb_all_word = np.load('one_emb_all.npz')
vecs_one_emb_all_word = one_emb_all_word['vectors']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_one_emb_all_word)} words")

# Random vector 
rand_vec_iw = np.random.randn(len(words_gg_iw), 10)
rand_vec_aw = np.random.randn(len(words), 10)

In [ ]:
# Important words
K = 50
K2 = 10
js_oei_iw = jaccard_overlap(
    vectors2=vecs_one_emb_intonation_important_word,
    vectors1=vecs_google_important_word,
    k=K,
    words=words_gg_iw
)
js_o_iw = jaccard_overlap(
    vectors2=vecs_original_important_word,
    vectors1=vecs_google_important_word,
    k=K,
    words=words_gg_iw
)

js_oe_iw = jaccard_overlap(
    vectors2=vecs_one_emb_important_word,
    vectors1=vecs_google_important_word,
    k=K,
    words=words_gg_iw
)

js_r_iw = jaccard_overlap(
    vectors2=rand_vec_iw,
    vectors1=vecs_google_important_word,
    k=K,
    words=words_gg_iw
)

# All words
js_oei_aw = jaccard_overlap(
    vectors2=vecs_one_emb_intonation_all_word,
    vectors1=vecs_google_all_word,
    k=K2,
    words=words
)
js_o_aw = jaccard_overlap(
    vectors2=vecs_original_all_word,
    vectors1=vecs_google_all_word,
    k=K2,
    words=words
)
js_oe_aw = jaccard_overlap(
    vectors2=vecs_one_emb_all_word,
    vectors1=vecs_google_all_word,
    k=K2,
    words=words
)

js_r_aw = jaccard_overlap(
    vectors2=rand_vec_aw,
    vectors1=vecs_google_all_word,
    k=K2,
    words=words
)

# Between model
js_oei_iw_by_o = jaccard_overlap(
    vectors2=vecs_one_emb_intonation_important_word,
    vectors1=vecs_original_important_word,
    k=K,
    words=words_gg_iw
)

js_oe_iw_by_o = jaccard_overlap(
    vectors2=vecs_one_emb_important_word,
    vectors1=vecs_original_important_word,
    k=K,
    words=words_gg_iw
)

js_oei_aw_by_o = jaccard_overlap(
    vectors2=vecs_one_emb_intonation_all_word,
    vectors1=vecs_original_all_word,
    k=K,
    words=words
)

js_oe_aw_by_o = jaccard_overlap(
    vectors2=vecs_one_emb_all_word,
    vectors1=vecs_original_all_word,
    k=K,
    words=words
)


In [ ]:
# verify len of jaccard score
print(f"data set ground trurh : {len(vecs_google_important_word)}")
print(f'One_Emb_Into_Imp : {len(js_oei_iw)}')
print(f'Original_Imp : {len(js_o_iw)}')
print(f"data set ground trurh all word : {len(vecs_google_all_word)}")
print(f'One_Emb_Into_All : {len(js_oei_aw)}')
print(f'Original_All : {len(js_o_aw)}')

In [ ]:
show_multi_plot_jaccard({
    "One_Emb_Into_Imp" : js_oei_iw,
    "Original_Imp" : js_o_iw,
    "One emb" : js_oe_iw,
    "Random" : js_r_iw,
    
    "One_Emb_Into_All" : js_oei_aw,
    "Original_All" : js_o_aw,
    "One emb All" : js_oe_aw,
    "Random 2" : js_r_aw,
    
    "Original and One emb into":js_oei_iw_by_o,
    "Original and One emb":js_oe_iw_by_o,
    "Original and One emb intp all word":js_oei_aw_by_o,
    "Original and One emb all word":js_oe_aw_by_o,
    
}, nb_cols=4)

In [ ]:
print(words_gg_iw)

In [ ]:
print(words_oei_iw)

## See specific word

In [ ]:
decoder_iw = {index : str(value) for index, value in enumerate(words)}
encoder_iw = {str(value) : index for index, value in enumerate(words)}

decoder_aw = {index : str(value) for index, value in enumerate(words)}
encoder_aw = {str(value) : index for index, value in enumerate(words)}

In [ ]:
print(encoder_aw.keys())

In [ ]:
print(js_oei_aw[encoder_aw["gorilla"]])
print(js_o_aw[encoder_aw["gorilla"]])
print(js_oe_aw[encoder_aw["gorilla"]])

In [ ]:
print(decoder_iw.keys())

In [ ]:
best_idx = np.argmax(js_oei_aw)
print(f"Best Aligned Word: '{decoder_aw[best_idx]}' (Score: {js_oei_aw[best_idx]:.2f})")
best_idx = int(np.argmax(js_o_aw))
print(f"Best Aligned Word: '{decoder_aw[best_idx]}' (Score: {js_o_aw[best_idx]:.2f})")
best_idx = int(np.argmax(js_oe_aw))
print(f"Best Aligned Word: '{decoder_aw[best_idx]}' (Score: {js_oe_aw[best_idx]:.2f})")

In [ ]:
top5pourcent = ['flashlight', 'click', 'keys', 'jingle', 'jangle', 'squeak', 'bicycle', 'tire', 'free', 'tiptoe', 'whoosh', 'banana', 'thump', 'bone', 'hyena', 'party', 'strong', 'shell', 'tap', 'parade', 'quietly', 'lawn', 'asleep', 'surprise', 'scuttle', 'cat', 'listen', 'blink', 'light', 'shocked', 'busted', 'march', 'clomp', 'shhh', 'clink', 'snooze', 'snoring', 'zzzz', 'peel']

In [ ]:
for w in top5pourcent:
    index = encoder_aw[w]
    print(f"score for word {w} ; {js_oei_aw[index]:.2f}")
    print(f"score for word {w} ; {js_o_aw[index]:.2f}")
    print(f"score for word {w} ; {js_oe_aw[index]:.2f}")

## See stability of model

## W2V with intonation (One embedding)

In [ ]:
instance_00 = np.load('embedding/seed_5_SGNS_OneEmbWeighted_.npz')
vec_instance_00 = instance_00['vectors']
word_instance_00 = instance_00['words']
print(f"For model : google_important_word , we have : {len(word_instance_00)} words")

instance_01 = np.load('embedding/seed_6_SGNS_OneEmbWeighted_.npz')
vec_instance_01 = instance_01['vectors']
word_instance_01 = instance_01['words']
print(f"For model : one_emb_intonation_important_word , we have : {len(word_instance_01)} words")

instance_02 = np.load('embedding/seed_7_SGNS_OneEmbWeighted_.npz')
vec_instance_02 = instance_02['vectors']
word_instance_02 = instance_02['words']
print(f"For model : one_emb_intonation_important_word , we have : {len(word_instance_02)} words")

instance_03 = np.load('embedding/seed_8_SGNS_OneEmbWeighted_.npz')
vec_instance_03 = instance_03['vectors']
word_instance_03 = instance_03['words']
print(f"For model : one_emb_intonation_important_word , we have : {len(word_instance_03)} words")

instance_04 = np.load('embedding/seed_9_SGNS_OneEmbWeighted_.npz')
vec_instance_04 = instance_04['vectors']
word_instance_04 = instance_04['words']
print(f"For model : google_important_word , we have : {len(word_instance_04)} words")

random = np.random.randn(len(word_instance_00), 10)

all_word_arrays = [
    word_instance_00, 
    word_instance_01, 
    word_instance_02, 
    word_instance_03, 
    word_instance_04,
]

mismatches = []
for i, words in enumerate(all_word_arrays[1:], start=1):
    if not np.array_equal(word_instance_00, words):
        mismatches.append(i)

if mismatches:
    print(f"ERROR: Words match failed for seeds: {mismatches}")
else:
    print("SUCCESS: All word arrays are identical.")
    
all_vec_arrays = [
    vec_instance_00,
    vec_instance_01,
    vec_instance_02,
    vec_instance_03,
    vec_instance_04,
    random
]

res_jaccard = {}

for ind1, vec1 in enumerate(all_vec_arrays):
    for ind2, vec2 in enumerate(all_vec_arrays):
        js = jaccard_overlap(vectors1=vec1, vectors2=vec2,
                            k=10, words=word_instance_00)
        res_jaccard[f"Instance n°{ind1} VS n°{ind2}"] = js
        
show_multi_plot_jaccard(res_jaccard, nb_cols=len(all_vec_arrays))

## One Emb

In [ ]:
instance_00 = np.load('embedding/seed_5_OnlyOneEmb_.npz')
vec_instance_00 = instance_00['vectors']
word_instance_00 = instance_00['words']
print(f"For model : google_important_word , we have : {len(word_instance_00)} words")

instance_01 = np.load('embedding/seed_6_OnlyOneEmb_.npz')
vec_instance_01 = instance_01['vectors']
word_instance_01 = instance_01['words']
print(f"For model : one_emb_intonation_important_word , we have : {len(word_instance_01)} words")

instance_02 = np.load('embedding/seed_7_OnlyOneEmb_.npz')
vec_instance_02 = instance_02['vectors']
word_instance_02 = instance_02['words']
print(f"For model : one_emb_intonation_important_word , we have : {len(word_instance_02)} words")

instance_03 = np.load('embedding/seed_8_OnlyOneEmb_.npz')
vec_instance_03 = instance_03['vectors']
word_instance_03 = instance_03['words']
print(f"For model : one_emb_intonation_important_word , we have : {len(word_instance_03)} words")

instance_04 = np.load('embedding/seed_9_OnlyOneEmb_.npz')
vec_instance_04 = instance_04['vectors']
word_instance_04 = instance_04['words']
print(f"For model : google_important_word , we have : {len(word_instance_04)} words")

random = np.random.randn(len(word_instance_00), 10)

all_word_arrays = [
    word_instance_00, 
    word_instance_01, 
    word_instance_02, 
    word_instance_03, 
    word_instance_04
]

mismatches = []
for i, words in enumerate(all_word_arrays[1:], start=1):
    if not np.array_equal(word_instance_00, words):
        mismatches.append(i)

if mismatches:
    print(f"ERROR: Words match failed for seeds: {mismatches}")
else:
    print("SUCCESS: All word arrays are identical.")
    
all_vec_arrays = [
    vec_instance_00,
    vec_instance_01,
    vec_instance_02,
    vec_instance_03,
    vec_instance_04,
    random
]

res_jaccard = {}

for ind1, vec1 in enumerate(all_vec_arrays):
    for ind2, vec2 in enumerate(all_vec_arrays):
        js = jaccard_overlap(vectors1=vec1, vectors2=vec2,
                            k=80, words=word_instance_00)
        res_jaccard[f"Instance n°{ind1} VS n°{ind2}"] = js
        
show_multi_plot_jaccard(res_jaccard, nb_cols=len(all_vec_arrays))

## Two embedding

In [ ]:
instance_00 = np.load('embedding/seed_5_SkipGramModel_.npz')
vec_instance_00 = instance_00['vectors']
word_instance_00 = instance_00['words']
print(f"For model : google_important_word , we have : {len(word_instance_00)} words")

instance_01 = np.load('embedding/seed_6_SkipGramModel_.npz')
vec_instance_01 = instance_01['vectors']
word_instance_01 = instance_01['words']
print(f"For model : one_emb_intonation_important_word , we have : {len(word_instance_01)} words")

instance_02 = np.load('embedding/seed_7_SkipGramModel_.npz')
vec_instance_02 = instance_02['vectors']
word_instance_02 = instance_02['words']
print(f"For model : one_emb_intonation_important_word , we have : {len(word_instance_02)} words")

instance_03 = np.load('embedding/seed_8_SkipGramModel_.npz')
vec_instance_03 = instance_03['vectors']
word_instance_03 = instance_03['words']
print(f"For model : one_emb_intonation_important_word , we have : {len(word_instance_03)} words")

instance_04 = np.load('embedding/seed_9_SkipGramModel_.npz')
vec_instance_04 = instance_04['vectors']
word_instance_04 = instance_04['words']
print(f"For model : google_important_word , we have : {len(word_instance_04)} words")

random = np.random.randn(len(word_instance_00), 10)

all_word_arrays = [
    word_instance_00, 
    word_instance_01, 
    word_instance_02, 
    word_instance_03, 
    word_instance_04
]

mismatches = []
for i, words in enumerate(all_word_arrays[1:], start=1):
    if not np.array_equal(word_instance_00, words):
        mismatches.append(i)

if mismatches:
    print(f"ERROR: Words match failed for seeds: {mismatches}")
else:
    print("SUCCESS: All word arrays are identical.")
    
all_vec_arrays = [
    vec_instance_00,
    vec_instance_01,
    vec_instance_02,
    vec_instance_03,
    vec_instance_04,
    random
]

res_jaccard = {}

for ind1, vec1 in enumerate(all_vec_arrays):
    for ind2, vec2 in enumerate(all_vec_arrays):
        js = jaccard_overlap(vectors1=vec1, vectors2=vec2,
                            k=10, words=word_instance_00)
        res_jaccard[f"Instance n°{ind1} VS n°{ind2}"] = js
        
show_multi_plot_jaccard(res_jaccard, nb_cols=len(all_vec_arrays))

# Mean Absolute Rank Error (MARE)

In [ ]:
def compute_mare(matrix_gold:np.ndarray, matrix_to_compare:np.ndarray, top_k:int=10)->float:
    """
    Docstring for compute_mare
    
    :param matrix_gold: Gold matrix of vector, is the reference 
    :type matrix_gold: np.ndarray
    :param matrix_to_compare: Vector matrix to be compared with the gold matrix
    :type matrix_to_compare: np.ndarray
    :param top_k: To avoid outlier, we not take all word because we have lot of noise word.
    :type top_k: int
    :return: Metric
    :rtype: float
    """
    
    norm_gold:np.ndarray = matrix_gold / np.linalg.norm(matrix_gold, axis=1, keepdims=True)
    norm_to_compare:np.ndarray = matrix_to_compare / np.linalg.norm(matrix_to_compare, axis=1, keepdims=True)
    
    sim_gold = np.dot(norm_gold, norm_gold.T) # Get matrix of similarity
    sim_to_compare = np.dot(norm_to_compare, norm_to_compare.T)
    
    ranks_gold = np.argsort(np.argsort(-sim_gold, axis=1), axis=1)
    ranks_to_compare = np.argsort(np.argsort(-sim_to_compare, axis=1), axis=1)
    
    top_gold_neighbors_indices = np.argsort(-sim_gold, axis=1)[:, 1:top_k+1]
    
    forward_errors = []
    rows = norm_to_compare.shape[0]
    
    for row_idx in range(rows):
        gold_neighbor_indices = top_gold_neighbors_indices[row_idx]
        r_gold = ranks_gold[row_idx, gold_neighbor_indices]
        r_gold = range(1, top_k +1)
        r_to_compare = ranks_to_compare[row_idx, gold_neighbor_indices]
        forward_errors.extend(np.abs(r_to_compare - r_gold))
    mare_fwd = np.mean(forward_errors)
    
    top_to_compare_neighbors_indices = np.argsort(-sim_to_compare, axis=1)[:, 1:top_k+1]
    backward_errors = []
    for i in range(rows):
        neighbor_indices = top_to_compare_neighbors_indices[i]
        # r_to_compare = ranks_to_compare[i, neighbor_indices]
        r_to_compare = range(1, top_k +1)
        
        r_gold = ranks_gold[i, neighbor_indices]

        backward_errors.extend(np.abs(r_to_compare - r_gold))
        
    mare_bwd = np.mean(backward_errors)
    
    
    return mare_fwd, mare_bwd

In [ ]:
# Important words
google_important_word = np.load('googleW2V/google_sub_embed.npz')
vecs_google_important_word = google_important_word['vectors']
words_gg_iw = google_important_word['words']
print(f"For model : google_important_word , we have : {len(vecs_google_important_word)} words")

one_emb_intonation_important_word = np.load('one_emb_into_imp.npz')
vecs_one_emb_intonation_important_word = one_emb_intonation_important_word['vectors']
words_oei_iw = one_emb_intonation_important_word['words']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_one_emb_intonation_important_word)} words")

original_important_word = np.load('original_imp.npz')
vecs_original_important_word = original_important_word['vectors']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_original_important_word)} words")

one_emb_important_word = np.load('one_emb_imp.npz')
vecs_one_emb_important_word = one_emb_important_word['vectors']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_one_emb_important_word)} words")

# All words
google_all_word = np.load('googleW2V/google_sub_embed_all_word.npz')
vecs_google_all_word = google_all_word['vectors']
words = google_all_word['words']
print(f"For model : google_important_word , we have : {len(vecs_google_all_word)} words")

one_emb_intonation_all_word = np.load('one_emb_into_all.npz')
vecs_one_emb_intonation_all_word = one_emb_intonation_all_word['vectors']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_one_emb_intonation_all_word)} words")

original_all_word = np.load('original_all.npz')
vecs_original_all_word = original_all_word['vectors']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_original_all_word)} words")

one_emb_all_word = np.load('one_emb_all.npz')
vecs_one_emb_all_word = one_emb_all_word['vectors']
print(f"For model : one_emb_intonation_important_word , we have : {len(vecs_one_emb_all_word)} words")

# Random vector 
rand_vec_iw = np.random.randn(len(words_gg_iw), 10)
rand_vec_aw = np.random.randn(len(words), 10)

In [ ]:
print(compute_mare(matrix_gold= vecs_google_important_word,
                   matrix_to_compare= vecs_one_emb_intonation_important_word,
                   top_k=10
                   ))

print(compute_mare(matrix_gold= vecs_google_important_word,
                   matrix_to_compare= vecs_original_important_word,
                   top_k=10
                   ))

print(compute_mare(matrix_gold= vecs_google_important_word,
                   matrix_to_compare= rand_vec_iw,
                   top_k=10
                   ))

# Manual cluster

## Function for manual cluster

In [3]:
def compute_distance_euclidien(vectors:np.ndarray, index_of_cluster:list[int], nb_outliers:int|None=100) -> dict:
    cluster_vectors = vectors[index_of_cluster]
    intra_distances = pdist(cluster_vectors, metric='euclidean')
    mean_intra_dist = np.mean(intra_distances)
    
    all_indices = np.arange(len(vectors))
    mask = np.isin(all_indices, index_of_cluster, invert=True)
    outside_indices = all_indices[mask]
    random_outside_indices = np.random.choice(outside_indices, size=nb_outliers, replace=False)
    outside_vectors = vectors[random_outside_indices]
    
    inter_distances = cdist(cluster_vectors, outside_vectors, metric='euclidean')
    mean_inter_dist = np.mean(inter_distances)


    return {
            "avg_dist_within_cluster": float(mean_intra_dist),
            "avg_dist_to_outsiders": float(mean_inter_dist),
            "ratio": float(mean_inter_dist / mean_intra_dist)
        }
    
def my_distance_euclidien(vectors:np.ndarray, index_of_cluster:list[int], nb_outliers:int|None=100) -> dict:
    result = {
        "no_traitement" : None,
        "PCA" : None,
        "norm" : None,
        "PCA and norm" : None,
    }
    
    result["no_traitement"] = compute_distance_euclidien(vectors, index_of_cluster, nb_outliers)
    
    pca = PCA(n_components=3)
    vectors_pca = pca.fit_transform(vectors)
    result["PCA"] = compute_distance_euclidien(vectors_pca, index_of_cluster, nb_outliers)
    
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1e-10 
    vectors_norm = vectors / norms
    result["norm"] = compute_distance_euclidien(vectors_norm, index_of_cluster, nb_outliers)
    
    norms_pca = np.linalg.norm(vectors_pca, axis=1, keepdims=True)
    norms_pca[norms_pca == 0] = 1e-10
    vectors_pca_norm = vectors_pca / norms_pca
    result["PCA and norm"] = compute_distance_euclidien(vectors_pca_norm, index_of_cluster, nb_outliers)

    return result

def compute_distance_rank_by_cluster(model_vectors:np.ndarray, cluster_index:list[int], n_random_samples=100):
    vectors = torch.tensor(model_vectors)
    norms = vectors.norm(dim=1, keepdim=True)
    vectors_norm = vectors / norms
    
    cluster_vecs = vectors_norm[cluster_index]
    sim_matrix = torch.mm(cluster_vecs, cluster_vecs.t()) # Dot product of normalized vectors = Cosine Similarity
    
    mask = torch.triu(
        torch.ones(len(cluster_index), len(cluster_index)), diagonal=1).bool() # Get upper triangle only (exclude diagonal 1.0s and duplicates)
    cluster_sim_mean = sim_matrix[mask].mean().item()

    num_vocab = len(model_vectors)
    idx_a = torch.randint(0, num_vocab, (n_random_samples,))
    idx_b = torch.randint(0, num_vocab, (n_random_samples,))
    
    vecs_a = vectors_norm[idx_a]
    vecs_b = vectors_norm[idx_b]

    global_sims = (vecs_a * vecs_b).sum(dim=1) # Dot product along dim 1
    
    global_mean = global_sims.mean().item()
    global_std = global_sims.std().item()
    
    z_score = (cluster_sim_mean - global_mean) / global_std # Z-Score
    
    return {
        "z_score": z_score,
        "raw_cluster_sim": cluster_sim_mean,
        "global_mean": global_mean
    }
    
def apply_metrics_with_clusters_in_vectors(
        paths_of_weight:list[str], 
        clusters:dict[str, list[str]],
        fct_metrics:list[Callable],
        n_random_samples=100
    ) -> dict:
    result = {
        i: {m.__name__: [] for m in fct_metrics} 
        for i in clusters.keys()
    }
    
    for path_w in paths_of_weight:
        name_file = path_w
        model = np.load(name_file)
        vecs = model['vectors']
        words = model['words']
        encoder = {str(w): i for i, w in enumerate(words)}
        
        for key, cl in clusters.items():
            valid_words = set(cl).intersection(encoder.keys())
            index_cluster = [encoder[w] for w in valid_words]
    
            for metric in fct_metrics:
                metric_name = metric.__name__
                metric_val = metric(vecs, index_cluster, n_random_samples)
                result[key][metric_name].append(metric_val)                
        
    return result


In [4]:
res = {x :  y for x, y in zip([1, 2, 4], [3, 3, 3])}
print(res)

{1: 3, 2: 3, 4: 3}


## All word

#### Define cluster and dico of result

In [4]:
cluster_animal = ["giraffe", "elephant", "mouse", "lion", "hyena", "armadillo", "gorilla"]
cluster_word_without_importance = ['there', 'is', 'the', 'he', 'has', 'a', 'to', 'you', 'my', 'and', 'that', 'right', 'him', 'doing', 'it', 'at', 'on', 'but', 'doesnot', 'does', 'his', 'with', 'very', 'else', 'itis', 'so', 'this', 'here', 'like', 'still', 'or', 'they', 'how', 'your', 'for', 'them', 'our', 'do', 'i', 'was', 'all', 'go', 'themselves', 'other', 'from', 'their', 'through', 'of', 'just', 'too', 'really', 'as', 'by', 'into', 'even', 'already', 'she', 'her', 'not', 'must', 'be', 'thatis', 'those', 'we', 'others', 'did', 'two', 'because', 'us']
all_result:dict = {
    "Our method with intonation" : None,
    "Our method without intonation" : None,
    "SGNS with intonation in our corpus" : None,
    "SGNS train in our corpus" : None,
    "SGNS pre-train" : None,
    "Random" : None,
}

### My method

In [5]:
list_of_path = [f"embedding/norm01/seed_{i}_SGNS_OneEmbWeighted_.npz" for i in range(0, 5)]

result = apply_metrics_with_clusters_in_vectors(
    paths_of_weight=list_of_path,
    clusters={
        "cluster_animal" : cluster_animal, 
        "cluster_word_without_importance" : cluster_word_without_importance,
    },
    fct_metrics=[compute_distance_rank_by_cluster, my_distance_euclidien],
    n_random_samples=330
)

print(result)
print(result["cluster_animal"]["compute_distance_rank_by_cluster"])
list_z_score = [zs["z_score"] for zs in result["cluster_animal"]["compute_distance_rank_by_cluster"]]
print(np.mean(list_z_score))
all_result["Our method with intonation"] = result

{'cluster_animal': {'compute_distance_rank_by_cluster': [{'z_score': 1.2540958904139077, 'raw_cluster_sim': 0.46461644768714905, 'global_mean': 0.03054111637175083}, {'z_score': 1.4392347002923829, 'raw_cluster_sim': 0.4880419373512268, 'global_mean': 0.003315998474135995}, {'z_score': 1.4179392733475518, 'raw_cluster_sim': 0.4840575158596039, 'global_mean': 0.020594222471117973}, {'z_score': 1.320018421669889, 'raw_cluster_sim': 0.4618180990219116, 'global_mean': 0.012165635824203491}, {'z_score': 1.5706874613734298, 'raw_cluster_sim': 0.5591071248054504, 'global_mean': 0.010890332981944084}], 'my_distance_euclidien': [{'no_traitement': {'avg_dist_within_cluster': 0.9004893262136131, 'avg_dist_to_outsiders': 1.55523005519432, 'ratio': 1.7270943807116153}, 'PCA': {'avg_dist_within_cluster': 0.28384811657746145, 'avg_dist_to_outsiders': 0.9707470367427181, 'ratio': 3.419952362015422}, 'norm': {'avg_dist_within_cluster': 1.0050668349296916, 'avg_dist_to_outsiders': 1.415979978506248, 'ra

### My model 2.0

In [6]:
list_of_path = [f"embedding/norm03/seed_{i}_SGNS_Weighted_.npz" for i in range(0, 5)]

result = apply_metrics_with_clusters_in_vectors(
    paths_of_weight=list_of_path,
    clusters={
        "cluster_animal" : cluster_animal, 
        "cluster_word_without_importance" : cluster_word_without_importance,
    },
    fct_metrics=[compute_distance_rank_by_cluster, my_distance_euclidien],
    n_random_samples=330
)

print(result)
print(result["cluster_animal"]["compute_distance_rank_by_cluster"])
list_z_score = [zs["z_score"] for zs in result["cluster_animal"]["compute_distance_rank_by_cluster"]]
print(np.mean(list_z_score))
all_result["SGNS with intonation in our corpus"] = result

{'cluster_animal': {'compute_distance_rank_by_cluster': [{'z_score': 0.5072311123035399, 'raw_cluster_sim': 0.3793153762817383, 'global_mean': 0.24623288214206696}, {'z_score': 0.36507760430190983, 'raw_cluster_sim': 0.3465469777584076, 'global_mean': 0.25072064995765686}, {'z_score': 0.585001463562771, 'raw_cluster_sim': 0.3819088637828827, 'global_mean': 0.22931227087974548}, {'z_score': 0.4697251500447194, 'raw_cluster_sim': 0.3859917223453522, 'global_mean': 0.2547188401222229}, {'z_score': 0.535292288593837, 'raw_cluster_sim': 0.37989768385887146, 'global_mean': 0.24445056915283203}], 'my_distance_euclidien': [{'no_traitement': {'avg_dist_within_cluster': 2.6670272272318765, 'avg_dist_to_outsiders': 4.01676972929234, 'ratio': 1.5060850104111496}, 'PCA': {'avg_dist_within_cluster': 1.7370901026409968, 'avg_dist_to_outsiders': 2.4763106947881184, 'ratio': 1.4255510931892608}, 'norm': {'avg_dist_within_cluster': 1.0939945469627899, 'avg_dist_to_outsiders': 1.2133632922294708, 'ratio'

### Model with only One emb

In [7]:
list_of_path = [f"embedding/noNorm/seed_{i}_OnlyOneEmb_.npz" for i in range(5, 10)]

result = apply_metrics_with_clusters_in_vectors(
    paths_of_weight=list_of_path,
    clusters={
        "cluster_animal" : cluster_animal, 
        "cluster_word_without_importance" : cluster_word_without_importance,
    },
    fct_metrics=[compute_distance_rank_by_cluster, my_distance_euclidien],
    n_random_samples=330
)

print(result)
print(result["cluster_animal"]["compute_distance_rank_by_cluster"])
list_z_score = [zs["z_score"] for zs in result["cluster_animal"]["compute_distance_rank_by_cluster"]]
print(np.mean(list_z_score))
all_result["Our method without intonation"] = result

{'cluster_animal': {'compute_distance_rank_by_cluster': [{'z_score': -0.003453694630303232, 'raw_cluster_sim': 0.07045622169971466, 'global_mean': 0.07161876559257507}, {'z_score': 0.2971513003648895, 'raw_cluster_sim': 0.15126678347587585, 'global_mean': 0.05757318437099457}, {'z_score': 0.021554253738008206, 'raw_cluster_sim': 0.0581430085003376, 'global_mean': 0.05131727457046509}, {'z_score': 0.39946727097169843, 'raw_cluster_sim': 0.1627933233976364, 'global_mean': 0.03769708052277565}, {'z_score': 0.18400610205106288, 'raw_cluster_sim': 0.1155126690864563, 'global_mean': 0.052972424775362015}], 'my_distance_euclidien': [{'no_traitement': {'avg_dist_within_cluster': 0.9190458464568985, 'avg_dist_to_outsiders': 1.1613713670346526, 'ratio': 1.2636707640994913}, 'PCA': {'avg_dist_within_cluster': 0.4847342498595634, 'avg_dist_to_outsiders': 0.6452073218500995, 'ratio': 1.3310537104341775}, 'norm': {'avg_dist_within_cluster': 1.337744775770814, 'avg_dist_to_outsiders': 1.4155106432818

### Model Original

In [8]:
list_of_path = [f"embedding/noNorm/seed_{i}_SkipGramModel_.npz" for i in range(5, 10)]

result = apply_metrics_with_clusters_in_vectors(
    paths_of_weight=list_of_path,
    clusters={
        "cluster_animal" : cluster_animal, 
        "cluster_word_without_importance" : cluster_word_without_importance,
    },
    fct_metrics=[compute_distance_rank_by_cluster, my_distance_euclidien],
    n_random_samples=330
)

print(result)
print(result["cluster_animal"]["compute_distance_rank_by_cluster"])
list_z_score = [zs["z_score"] for zs in result["cluster_animal"]["compute_distance_rank_by_cluster"]]
print(np.mean(list_z_score))
print(np.std(list_z_score))
all_result["SGNS train in our corpus"] = result

{'cluster_animal': {'compute_distance_rank_by_cluster': [{'z_score': 0.6079232144943069, 'raw_cluster_sim': 0.40149542689323425, 'global_mean': 0.23755468428134918}, {'z_score': 0.7133055311666546, 'raw_cluster_sim': 0.41423630714416504, 'global_mean': 0.2341473251581192}, {'z_score': 0.6200556252118892, 'raw_cluster_sim': 0.3953515589237213, 'global_mean': 0.23003195226192474}, {'z_score': 0.27841179999252436, 'raw_cluster_sim': 0.32360532879829407, 'global_mean': 0.25079476833343506}, {'z_score': 0.4095790484932844, 'raw_cluster_sim': 0.34291043877601624, 'global_mean': 0.23551996052265167}], 'my_distance_euclidien': [{'no_traitement': {'avg_dist_within_cluster': 2.430586527845447, 'avg_dist_to_outsiders': 4.135580717583288, 'ratio': 1.701474384970448}, 'PCA': {'avg_dist_within_cluster': 1.2778282837592359, 'avg_dist_to_outsiders': 2.421075464067603, 'ratio': 1.8946798210985398}, 'norm': {'avg_dist_within_cluster': 1.0680300889503682, 'avg_dist_to_outsiders': 1.2024156409616662, 'rat

### Modèle pre-train

In [9]:
list_of_path = [f"googleW2V/google_sub_embed_all_word.npz"]

result = apply_metrics_with_clusters_in_vectors(
    paths_of_weight=list_of_path,
    clusters={
        "cluster_animal" : cluster_animal, 
        "cluster_word_without_importance" : cluster_word_without_importance,
    },
    fct_metrics=[compute_distance_rank_by_cluster, my_distance_euclidien],
    n_random_samples=330
)

print(result)
print(result["cluster_animal"]["compute_distance_rank_by_cluster"])
list_z_score = [zs["z_score"] for zs in result["cluster_animal"]["compute_distance_rank_by_cluster"]]
print(np.mean(list_z_score))
print(np.std(list_z_score))
all_result["SGNS pre-train"] = result

{'cluster_animal': {'compute_distance_rank_by_cluster': [{'z_score': 3.0086300606443817, 'raw_cluster_sim': 0.40908491611480713, 'global_mean': 0.12845000624656677}], 'my_distance_euclidien': [{'no_traitement': {'avg_dist_within_cluster': 3.5852674242561076, 'avg_dist_to_outsiders': 4.035703579541037, 'ratio': 1.125635301912908}, 'PCA': {'avg_dist_within_cluster': 0.5818474307140886, 'avg_dist_to_outsiders': 1.6459743309929868, 'ratio': 2.828876169433143}, 'norm': {'avg_dist_within_cluster': 1.0803801413114655, 'avg_dist_to_outsiders': 1.348567910523654, 'ratio': 1.2482346342340553}, 'PCA and norm': {'avg_dist_within_cluster': 0.2626443034991377, 'avg_dist_to_outsiders': 1.3843671178596852, 'ratio': 5.270881947242501}}]}, 'cluster_word_without_importance': {'compute_distance_rank_by_cluster': [{'z_score': 1.4686494518063489, 'raw_cluster_sim': 0.25375884771347046, 'global_mean': 0.11863735318183899}], 'my_distance_euclidien': [{'no_traitement': {'avg_dist_within_cluster': 2.41039572550

### random

In [10]:
nb_instance = 50

In [11]:
name_file = f"embedding/norm01/seed_{0}_SGNS_OneEmbWeighted_.npz"
model = np.load(name_file)
words = model['words']
for instance in range(nb_instance):
    vectors = np.random.randn(len(words), 10)
    np.savez(
        f'embedding/rand/{instance}_.npz', 
        vectors=vectors,
        words=words
    )

In [12]:
list_of_path = [f"embedding/rand/{instance}_.npz" for instance in range(nb_instance)]

result = apply_metrics_with_clusters_in_vectors(
    paths_of_weight=list_of_path,
    clusters={
        "cluster_animal" : cluster_animal, 
        "cluster_word_without_importance" : cluster_word_without_importance,
    },
    fct_metrics=[compute_distance_rank_by_cluster, my_distance_euclidien],
    n_random_samples=330
)

print(result)
print(result["cluster_animal"]["compute_distance_rank_by_cluster"])
list_z_score = [zs["z_score"] for zs in result["cluster_animal"]["compute_distance_rank_by_cluster"]]
print(np.mean(list_z_score))
print(np.std(list_z_score))
all_result["Random"] = result

{'cluster_animal': {'compute_distance_rank_by_cluster': [{'z_score': -0.13561170677156895, 'raw_cluster_sim': -0.02077999160751425, 'global_mean': 0.020375374477267665}, {'z_score': -0.21305063009546202, 'raw_cluster_sim': -0.07026597360198965, 'global_mean': -0.0012638189088732571}, {'z_score': -0.12433943160690532, 'raw_cluster_sim': -0.05103506194079991, 'global_mean': -0.011556058000391868}, {'z_score': -0.3722585309798137, 'raw_cluster_sim': -0.09661112041281908, 'global_mean': 0.027098681970054914}, {'z_score': 0.0801589203352669, 'raw_cluster_sim': -0.005526454250354208, 'global_mean': -0.03160582623802829}, {'z_score': 0.002140784661202944, 'raw_cluster_sim': -0.030879737795256646, 'global_mean': -0.0315336535579286}, {'z_score': -0.24388732213484332, 'raw_cluster_sim': -0.07758549281219104, 'global_mean': -0.002385886201746465}, {'z_score': 0.3028756215264662, 'raw_cluster_sim': 0.0758348766802108, 'global_mean': -0.02157970247467596}, {'z_score': -0.21181935255969603, 'raw_cl

## Visu

### Distance euclidien

In [ ]:
models = list(all_result.keys())
result_all_distance_euclidien = {}
for m in models : 
    result_all_distance_euclidien[m] = all_result[m]['cluster_animal']["my_distance_euclidien"]
        
list_z_score_mean = []
list_raw_cluster_sim_mean = []
list_global_mean_mean = []
list_PCA_norm_mean = []

list_z_score_std = []
list_raw_cluster_sim_std = []
list_global_mean_std = []
list_PCA_norm_std = []

for model, metric_by_instance in result_all_distance_euclidien.items():
    print(model)
    print(metric_by_instance)
    l_z_score = []
    l_raw_cluster_sim = []
    l_global_mean = []
    l_pca_norm = []
    for type_distance in metric_by_instance:
        l_z_score.append(type_distance["no_traitement"]["ratio"])
        l_raw_cluster_sim.append(type_distance["norm"]["ratio"])
        l_global_mean.append(type_distance["PCA"]["ratio"])
        l_pca_norm.append(type_distance["PCA and norm"]["ratio"])
        
    list_z_score_mean.append(np.mean(l_z_score))
    list_raw_cluster_sim_mean.append(np.mean(l_raw_cluster_sim))
    list_global_mean_mean.append(np.mean(l_global_mean))
    list_PCA_norm_mean.append(np.mean(l_pca_norm))
    
    list_z_score_std.append(np.std(l_z_score))
    list_raw_cluster_sim_std.append(np.std(l_raw_cluster_sim))
    list_global_mean_std.append(np.std(l_global_mean))
    list_PCA_norm_std.append(np.std(l_pca_norm))
    
x = np.arange(len(models))  # label locations
width = 0.1                # width of the bars

fig, ax = plt.subplots(figsize=(10, 6))

# Plot bars
rects1 = ax.bar(x - width * 2, list_z_score_mean, width, yerr=list_z_score_std, 
                label='no traitement', capsize=5, color='skyblue', edgecolor='black')
rects2 = ax.bar(x - width/2, list_raw_cluster_sim_mean, width, yerr=list_raw_cluster_sim_std, 
                label='norm', capsize=5, color='orange', edgecolor='black')
rects3 = ax.bar(x + width/2, list_global_mean_mean, width, yerr=list_raw_cluster_sim_std, 
                label='pca', capsize=5, color='green', edgecolor='black')
rects4 = ax.bar(x + width *2, list_PCA_norm_mean, width, yerr=list_PCA_norm_std, 
                label='pca norm', capsize=5, color='red', edgecolor='black')

ax.bar_label(rects1, padding=3, fmt='%.2f')
ax.bar_label(rects2, padding=3, fmt='%.2f')
ax.bar_label(rects3, padding=3, fmt='%.2f')
ax.bar_label(rects4, padding=3, fmt='%.2f')

# Add labels, title and custom x-axis tick labels
ax.set_ylabel('Mean Score')
ax.set_title('Comparison of PCA vs noTraitement across Models in cluster animal')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15)
ax.legend()

plt.tight_layout()
plt.savefig('model_comparison_histogram.png')
plt.show()
    

In [ ]:
models = list(all_result.keys())
result_all_distance_euclidien = {}
for m in models : 
    result_all_distance_euclidien[m] = all_result[m]['cluster_word_without_importance']["my_distance_euclidien"]
        
list_z_score_mean = []
list_raw_cluster_sim_mean = []
list_global_mean_mean = []
list_PCA_norm_mean = []

list_z_score_std = []
list_raw_cluster_sim_std = []
list_global_mean_std = []
list_PCA_norm_std = []

for model, metric_by_instance in result_all_distance_euclidien.items():
    print(model)
    print(metric_by_instance)
    l_z_score = []
    l_raw_cluster_sim = []
    l_global_mean = []
    l_pca_norm = []
    for type_distance in metric_by_instance:
        l_z_score.append(type_distance["no_traitement"]["ratio"])
        l_raw_cluster_sim.append(type_distance["norm"]["ratio"])
        l_global_mean.append(type_distance["PCA"]["ratio"])
        l_pca_norm.append(type_distance["PCA and norm"]["ratio"])
        
    list_z_score_mean.append(np.mean(l_z_score))
    list_raw_cluster_sim_mean.append(np.mean(l_raw_cluster_sim))
    list_global_mean_mean.append(np.mean(l_global_mean))
    list_PCA_norm_mean.append(np.mean(l_pca_norm))
    
    list_z_score_std.append(np.std(l_z_score))
    list_raw_cluster_sim_std.append(np.std(l_raw_cluster_sim))
    list_global_mean_std.append(np.std(l_global_mean))
    list_PCA_norm_std.append(np.std(l_pca_norm))
    
x = np.arange(len(models))  # label locations
width = 0.1                # width of the bars

fig, ax = plt.subplots(figsize=(10, 6))

# Plot bars
rects1 = ax.bar(x - width * 2, list_z_score_mean, width, yerr=list_z_score_std, 
                label='no traitement', capsize=5, color='skyblue', edgecolor='black')
rects2 = ax.bar(x - width/2, list_raw_cluster_sim_mean, width, yerr=list_raw_cluster_sim_std, 
                label='norm', capsize=5, color='orange', edgecolor='black')
rects3 = ax.bar(x + width/2, list_global_mean_mean, width, yerr=list_raw_cluster_sim_std, 
                label='PCA', capsize=5, color='green', edgecolor='black')
rects4 = ax.bar(x + width *2, list_PCA_norm_mean, width, yerr=list_PCA_norm_std, 
                label='pca norm', capsize=5, color='red', edgecolor='black')

ax.bar_label(rects1, padding=3, fmt='%.2f')
ax.bar_label(rects2, padding=3, fmt='%.2f')
ax.bar_label(rects3, padding=3, fmt='%.2f')
ax.bar_label(rects4, padding=3, fmt='%.2f')

# Add labels, title and custom x-axis tick labels
ax.set_ylabel('Mean Score')
ax.set_title('Comparison of PCA vs noTraitement across Models in cluster bad word')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15)
ax.legend()

plt.tight_layout()
plt.savefig('model_comparison_histogram.png')
plt.show()
    

### visu z_score

In [ ]:
models = list(all_result.keys())
result_all_distance_euclidien = {}
for m in models : 
    result_all_distance_euclidien[m] = all_result[m]['cluster_animal']["compute_distance_rank_by_cluster"]
        
list_z_score_mean = []
list_raw_cluster_sim_mean = []
list_global_mean_mean = []

list_z_score_std = []
list_raw_cluster_sim_std = []
list_global_mean_std = []

list_z_score = []

for model, metric_by_instance in result_all_distance_euclidien.items():
    l_z_score = []
    l_raw_cluster_sim = []
    l_global_mean = []
    for type_distance in metric_by_instance:
        l_z_score.append(type_distance["z_score"])
        l_raw_cluster_sim.append(type_distance["raw_cluster_sim"])
        l_global_mean.append(type_distance["global_mean"])
        
    list_z_score_mean.append(np.mean(l_z_score))
    list_raw_cluster_sim_mean.append(np.mean(l_raw_cluster_sim))
    list_global_mean_mean.append(np.mean(l_global_mean))
    
    list_z_score_std.append(np.std(l_z_score))
    list_raw_cluster_sim_std.append(np.std(l_raw_cluster_sim))
    list_global_mean_std.append(np.std(l_global_mean))
    
    list_z_score.append(l_z_score)
    
x = np.arange(len(models))  # label locations
width = 0.25                # width of the bars

fig, ax = plt.subplots(figsize=(10, 6))

# Plot bars
rects1 = ax.bar(x - width, list_z_score_mean, width, yerr=list_z_score_std, 
                label='z_score', capsize=5, color='skyblue', edgecolor='black')
rects2 = ax.bar(x , list_raw_cluster_sim_mean, width, yerr=list_raw_cluster_sim_std, 
                label='raw_cluster_sim', capsize=5, color='orange', edgecolor='black')
rects3 = ax.bar(x + width, list_global_mean_mean, width, yerr=list_global_mean_std, 
                label='global_means', capsize=5, color='green', edgecolor='black')

ax.bar_label(rects1, padding=3, fmt='%.2f')
ax.bar_label(rects2, padding=3, fmt='%.2f')
ax.bar_label(rects3, padding=3, fmt='%.2f')

# Add labels, title and custom x-axis tick labels
ax.set_ylabel('Mean Score')
ax.set_title('Comparaison of different model using Z score and cluster "cluster_animal"')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15)
ax.legend()

plt.tight_layout()
plt.savefig('model_comparison_histogram.png')
plt.show()
plt.show()

anova = f_oneway(list_z_score[0],list_z_score[1],list_z_score[2],list_z_score[3],list_z_score[4],list_z_score[5])
print(anova)
    

In [ ]:
models = list(all_result.keys())
result_all_distance_euclidien = {}
for m in models : 
    result_all_distance_euclidien[m] = all_result[m]['cluster_word_without_importance']["compute_distance_rank_by_cluster"]
        
list_z_score_mean = []
list_raw_cluster_sim_mean = []
list_global_mean_mean = []

list_z_score_std = []
list_raw_cluster_sim_std = []
list_global_mean_std = []

list_z_score = []
list_raw_cluster_sim = []
list_global_mean = []

for model, metric_by_instance in result_all_distance_euclidien.items():
    l_z_score = []
    l_raw_cluster_sim = []
    l_global_mean = []
    for type_distance in metric_by_instance:
        l_z_score.append(type_distance["z_score"])
        l_raw_cluster_sim.append(type_distance["raw_cluster_sim"])
        l_global_mean.append(type_distance["global_mean"])
        
    list_z_score_mean.append(np.mean(l_z_score))
    list_raw_cluster_sim_mean.append(np.mean(l_raw_cluster_sim))
    list_global_mean_mean.append(np.mean(l_global_mean))
    
    list_z_score_std.append(np.std(l_z_score))
    list_raw_cluster_sim_std.append(np.std(l_raw_cluster_sim))
    list_global_mean_std.append(np.std(l_global_mean))
    list_z_score_mean
    list_z_score.append(l_z_score)
    list_raw_cluster_sim.append(l_raw_cluster_sim)
    list_global_mean.append(l_global_mean)
        
x = np.arange(len(models))  # label locations
width = 0.25                # width of the bars

fig, ax = plt.subplots(figsize=(10, 6))

# Plot bars
rects1 = ax.bar(x - width, list_z_score_mean, width, yerr=list_z_score_std, 
                label='z_score', capsize=5, color='skyblue', edgecolor='black')
rects2 = ax.bar(x , list_raw_cluster_sim_mean, width, yerr=list_raw_cluster_sim_std, 
                label='raw_cluster_sim', capsize=5, color='orange', edgecolor='black')
rects3 = ax.bar(x + width, list_global_mean_mean, width, yerr=list_global_mean_std, 
                label='global_means', capsize=5, color='green', edgecolor='black')

ax.bar_label(rects1, padding=3, fmt='%.2f')
ax.bar_label(rects2, padding=3, fmt='%.2f')
ax.bar_label(rects3, padding=3, fmt='%.2f')

# Add labels, title and custom x-axis tick labels
ax.set_ylabel('Mean Score')
ax.set_title('Comparaison of different model using Z score and cluster "bad wrod"')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15)
ax.legend()


plt.tight_layout()
plt.savefig('model_comparison_histogram.png')
plt.show()
plt.show()


anova = f_oneway(list_z_score[0],list_z_score[1],list_z_score[2],list_z_score[3],list_z_score[4],list_z_score[5])
print(anova)



# Visu embedding

In [5]:
cluster_animal = ["giraffe", "elephant", "mouse", "lion", "hyena", "armadillo", "gorilla"]
cluster_word_without_importance = ['there', 'is', 'the', 'he', 'has', 'a', 'to', 'you', 'my', 'and', 'that', 'right', 'him', 'doing', 'it', 'at', 'on', 'but', 'doesnot', 'does', 'his', 'with', 'very', 'else', 'itis', 'so', 'this', 'here', 'like', 'still', 'or', 'they', 'how', 'your', 'for', 'them', 'our', 'do', 'i', 'was', 'all', 'go', 'themselves', 'other', 'from', 'their', 'through', 'of', 'just', 'too', 'really', 'as', 'by', 'into', 'even', 'already', 'she', 'her', 'not', 'must', 'be', 'thatis', 'those', 'we', 'others', 'did', 'two', 'because', 'us']

## SGNS de base

In [ ]:
name_file = f"embedding/noNorm/seed_{5}_SkipGramModel_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}
    
base_colors = {
    'banana': ("yellow",  "lightyellow"),
    'gorilla': ("gray", "lightgray"),
    'zookeeper': ("brown", "sandybrown"),
    "little": ("pink", "lightpink"),
    "yellow": ("gold", "lightgoldenrodyellow"),
    'light' : ("gold", "lightgoldenrodyellow"),
    'turned': ('limegreen', 'green'),
    'spot': ('beige', 'bisque'),
    'finding': ('red', 'firebrick'),
}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
    
fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_animal,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_word_without_importance,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

## Different norm

In [ ]:
name_file = f"embedding/norm01/seed_0_SGNS_OneEmbWeighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}
    
base_colors = {
    'banana': ("yellow",  "lightyellow"),
    'gorilla': ("gray", "lightgray"),
    'zookeeper': ("brown", "sandybrown"),
    "little": ("pink", "lightpink"),
    "yellow": ("gold", "lightgoldenrodyellow"),
    'light' : ("gold", "lightgoldenrodyellow"),
    'turned': ('limegreen', 'green'),
    'spot': ('beige', 'bisque'),
    'finding': ('red', 'firebrick'),
}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
    
fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_animal,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_word_without_importance,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

In [ ]:
name_file = f"embedding/norm02/seed_0_SGNS_OneEmbWeighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}
    
base_colors = {
    'banana': ("yellow",  "lightyellow"),
    'gorilla': ("gray", "lightgray"),
    'zookeeper': ("brown", "sandybrown"),
    "little": ("pink", "lightpink"),
    "yellow": ("gold", "lightgoldenrodyellow"),
    'light' : ("gold", "lightgoldenrodyellow"),
    'turned': ('limegreen', 'green'),
    'spot': ('beige', 'bisque'),
    'finding': ('red', 'firebrick'),
}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
    
fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_animal,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_word_without_importance,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

In [ ]:
name_file = f"embedding/norm03/seed_0_SGNS_OneEmbWeighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}
    
base_colors = {
    'banana': ("yellow",  "lightyellow"),
    'gorilla': ("gray", "lightgray"),
    'zookeeper': ("brown", "sandybrown"),
    "little": ("pink", "lightpink"),
    "yellow": ("gold", "lightgoldenrodyellow"),
    'light' : ("gold", "lightgoldenrodyellow"),
    'turned': ('limegreen', 'green'),
    'spot': ('beige', 'bisque'),
    'finding': ('red', 'firebrick'),
}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
    
fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_animal,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_word_without_importance,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

## Two emb weighted

In [ ]:
name_file = f"embedding/norm03/seed_0_SGNS_Weighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}
    
base_colors = {
    'banana': ("yellow",  "lightyellow"),
    'gorilla': ("gray", "lightgray"),
    'zookeeper': ("brown", "sandybrown"),
    "little": ("pink", "lightpink"),
    "yellow": ("gold", "lightgoldenrodyellow"),
    'light' : ("gold", "lightgoldenrodyellow"),
    'turned': ('limegreen', 'green'),
    'spot': ('beige', 'bisque'),
    'finding': ('red', 'firebrick'),
}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
    
fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_animal,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_word_without_importance,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

In [ ]:
name_file = f"embedding/norm02/seed_0_SGNS_Weighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']
decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}
    
base_colors = {
    'banana': ("yellow",  "lightyellow"),
    'gorilla': ("gray", "lightgray"),
    'zookeeper': ("brown", "sandybrown"),
    "little": ("pink", "lightpink"),
    "yellow": ("gold", "lightgoldenrodyellow"),
    'light' : ("gold", "lightgoldenrodyellow"),
    'turned': ('limegreen', 'green'),
    'spot': ('beige', 'bisque'),
    'finding': ('red', 'firebrick'),
}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
    
fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_animal,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_word_without_importance,
    nb_neighbors=1, base_color=base_colors
)

fig.show()


## See cluster by intonation

In [21]:
name_file = f"embedding/norm02/seed_0_SGNS_Weighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}

print(words[:10])
print(cluster_word_without_importance[:10])
fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=words,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

['look' 'there' 'is' 'the' 'zookeeper' 'he' 'has' 'a' 'big' 'flashlight']
['there', 'is', 'the', 'he', 'has', 'a', 'to', 'you', 'my', 'and']


In [6]:
data_intonation = pd.read_csv("data/intonation_word_GNG.csv", index_col=0)
print(data_intonation)

base_colors = {
    'banana': ("yellow",  "lightyellow"),
    'gorilla': ("gray", "lightgray"),
    'zookeeper': ("brown", "sandybrown"),
    "little": ("pink", "lightpink"),
    "yellow": ("gold", "lightgoldenrodyellow"),
    'light' : ("gold", "lightgoldenrodyellow"),
    'turned': ('limegreen', 'green'),
    'spot': ('beige', 'bisque'),
    'finding': ('red', 'firebrick'),
}

for t in data_intonation["range"].to_dict().items():
    if t[1] == '0-1':
        base_colors[t[0]] = "black"
    if t[1] == '1-2':
        base_colors[t[0]] = "orange"
    if t[1] == '2-3':
        base_colors[t[0]] = "blue"
    if t[1] == '3-4':
        base_colors[t[0]] = "green"
    if t[1] == '4-5':
        base_colors[t[0]] = "red"
print(base_colors)


           intonation_mean range
look              2.000000   1-2
there             0.000000   0-1
is                0.017094   0-1
the               0.000000   0-1
zookeeper         4.160000   4-5
...                    ...   ...
midnight          3.000000   2-3
outside           2.000000   1-2
moon              4.000000   3-4
stars             4.000000   3-4
shining           3.000000   2-3

[410 rows x 2 columns]
{'banana': 'red', 'gorilla': 'red', 'zookeeper': 'red', 'little': 'blue', 'yellow': 'blue', 'light': 'red', 'turned': 'orange', 'spot': 'blue', 'finding': 'orange', 'look': 'orange', 'there': 'black', 'is': 'black', 'the': 'black', 'he': 'black', 'has': 'black', 'a': 'black', 'big': 'green', 'flashlight': 'red', 'to': 'black', 'see': 'orange', 'in': 'black', 'dark': 'green', 'click': 'red', 'what': 'orange', 'saying': 'orange', 'animal': 'blue', 'says': 'orange', 'good': 'red', 'night': 'red', 'can': 'orange', 'you': 'black', 'say': 'orange', 'oh': 'green', 'my': 'black', '

### fct

In [7]:
def components_to_fig_3D_simple(
    components: np.ndarray,
    encoder: dict,
    base_color: dict = None,
    default_color: str = "lightgray",
    title: str = "3D Representation of Vectors",
    _min: Optional[float] = None,
    _max: Optional[float] = None,
) -> go.Figure:
    """
    Plots all vectors in 3D without neighbor logic. Points are colored based on 
    base_color mapping. Labels are only visible on hover.
    
    base_color should be a dictionary mapping words to CSS colors: e.g., {"apple": "red"}
    """
    if base_color is None:
        base_color = {}

    # 1. Map rows in the component array back to their words
    # This guarantees the order of colors matches the rows in 'components'
    idx_to_word = {idx: word for word, idx in encoder.items()}
    words_ordered = [idx_to_word.get(i, f"vec_{i}") for i in range(components.shape[0])]

    # 2. Extract coordinates
    xs = components[:, 0]
    ys = components[:, 1]
    zs = components[:, 2]

    # 3. Create a color list for every single point
    colors = [base_color.get(w, default_color) for w in words_ordered]

    # 4. Handle axes limits
    if _min is None or _max is None:
        max_abs_val = np.max(np.abs(components))
        limit = max_abs_val * 1.1

        if _min is None:
            _min = -limit
        if _max is None:
            _max = limit

    # 5. Initialize figure and add the main trace
    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=xs,
            y=ys,
            z=zs,
            mode="markers", # "markers" only, removes text from the canvas
            marker=dict(
                size=4, 
                color=colors, 
                opacity=0.8
            ),
            hovertext=words_ordered,
            hoverinfo="text", # Show only the word when hovering
            name="Vectors"
        )
    )

    # 6. Draw custom axes and arrows (kept from your original code)
    max_range = np.max(np.ptp(components, axis=0))
    if max_range == 0:
        max_range = 1.0
    scale = max_range * 0.05

    axes_traces = [
        go.Scatter3d(x=[-scale, scale], y=[0, 0], z=[0, 0], mode="lines", line=dict(color="red", width=4), name="axis X"),
        go.Scatter3d(x=[0, 0], y=[-scale, scale], z=[0, 0], mode="lines", line=dict(color="green", width=4), name="axis Y"),
        go.Scatter3d(x=[0, 0], y=[0, 0], z=[-scale, scale], mode="lines", line=dict(color="blue", width=4), name="axis Z"),
    ]

    arrow_len = scale * 0.008
    arrow_traces = [
        go.Scatter3d(x=[scale, scale - arrow_len], y=[0, 0], z=[0, 0], mode="lines", line=dict(color="red", width=6), showlegend=False),
        go.Scatter3d(x=[0, 0], y=[scale, scale - arrow_len], z=[0, 0], mode="lines", line=dict(color="green", width=6), showlegend=False),
        go.Scatter3d(x=[0, 0], y=[0, 0], z=[scale, scale - arrow_len], mode="lines", line=dict(color="blue", width=6), showlegend=False)
    ]
    
    labels_traces = [
        go.Scatter3d(x=[scale], y=[0], z=[0], mode="text", text=["X"], textposition="top center", showlegend=False),
        go.Scatter3d(x=[0], y=[scale], z=[0], mode="text", text=["Y"], textposition="top center", showlegend=False),
        go.Scatter3d(x=[0], y=[0], z=[scale], mode="text", text=["Z"], textposition="top center", showlegend=False)
    ]

    for t in axes_traces + arrow_traces + labels_traces:
        fig.add_trace(t)

    # 7. Final Layout settings
    fig.update_layout(
        title=title,
        width=1000, 
        height=800,
        scene=dict(
            xaxis=dict(nticks=4, range=[_min, _max], title="PC1"),
            yaxis=dict(nticks=4, range=[_min, _max], title="PC2"),
            zaxis=dict(nticks=4, range=[_min, _max], title="PC3"),
        ),
        scene_aspectmode="cube",
    )
    
    return fig

def components_to_fig_mollweide(
    components: np.ndarray,
    encoder: dict,
    base_color: dict = None,
    default_color: str = "lightgray",
    title: str = "2D Mollweide Projection of Vectors",
) -> go.Figure:
    """
    Plots all vectors in a 2D Mollweide projection. 
    Points are colored based on base_color mapping. Labels are visible on hover.
    """
    if base_color is None:
        base_color = {}

    # 1. Map rows in the component array back to their words
    idx_to_word = {idx: word for word, idx in encoder.items()}
    words_ordered = [idx_to_word.get(i, f"vec_{i}") for i in range(components.shape[0])]

    # 2. Extract coordinates
    x = components[:, 0]
    y = components[:, 1]
    z = components[:, 2]

    # 3. Calculate Radius, Azimuth, and Elevation
    r = np.sqrt(x**2 + y**2 + z**2)
    # Prevent division by zero if there's a vector exactly at the origin (0,0,0)
    r[r == 0] = 1e-10 
    
    azimuth_rad = np.arctan2(y, x)
    elevation_rad = np.arcsin(z / r)

    # 4. Convert to Degrees for Plotly Scattergeo (Longitude & Latitude)
    lon_deg = np.degrees(azimuth_rad)
    lat_deg = np.degrees(elevation_rad)

    # 5. Create a color list for every single point
    colors = [base_color.get(w, default_color) for w in words_ordered]

    # 6. Initialize figure with Scattergeo
    fig = go.Figure()

    fig.add_trace(
        go.Scattergeo(
            lon=lon_deg,
            lat=lat_deg,
            mode="markers",
            marker=dict(
                size=9, 
                color=colors, 
                opacity=1,
                line=dict(width=0.5, color='white') # Helps separate overlapping dots
            ),
            hovertext=words_ordered,
            hoverinfo="text",
            name="Projected Vectors"
        )
    )

    # 7. Layout settings for Mollweide
    fig.update_layout(
        title=title,
        width=1000, 
        height=600,
        geo=dict(
            projection_type="mollweide",
            showcoastlines=False, # We don't want Earth coastlines on a math plot
            showland=False,
            showocean=False,
            showframe=True,       # Shows the outer oval shape
            # showgrid=True,        # Shows the longitude/latitude curve lines
            # gridcolor="lightgray"
        )
    )
    
    return fig

### fig

In [8]:
name_file = f"embedding/norm02/seed_0_SGNS_Weighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}

print(words[:10])
print(cluster_word_without_importance[:10])
fig = components_to_fig_3D_simple(
    components=emb_pca,
    encoder=encoder,
    base_color=base_colors
)

fig.show()

['look' 'there' 'is' 'the' 'zookeeper' 'he' 'has' 'a' 'big' 'flashlight']
['there', 'is', 'the', 'he', 'has', 'a', 'to', 'you', 'my', 'and']


NameError: name 'emb_pca' is not defined

In [ ]:
name_file = f"embedding/norm01/seed_0_SGNS_OneEmbWeighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
fig = components_to_fig_3D_simple(
    components=emb_pca,
    encoder=encoder,
    base_color=base_colors
)

fig.show()

name_file = f"embedding/norm02/seed_0_SGNS_OneEmbWeighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
fig = components_to_fig_3D_simple(
    components=emb_pca,
    encoder=encoder,
    base_color=base_colors
)

fig.show()

name_file = f"embedding/norm03/seed_0_SGNS_OneEmbWeighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
fig = components_to_fig_3D_simple(
    components=emb_pca,
    encoder=encoder,
    base_color=base_colors
)

fig.show()

In [16]:
name_file = f"embedding/norm01/seed_0_SGNS_OneEmbWeighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
norms = np.linalg.norm(emb_pca, axis=1, keepdims=True)
norms[norms == 0] = 1e-10 
vectors_norm = emb_pca / norms
fig = components_to_fig_3D_simple(
    components=vectors_norm,
    encoder=encoder,
    base_color=base_colors
)

fig.show()

name_file = f"embedding/norm02/seed_0_SGNS_OneEmbWeighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
norms = np.linalg.norm(emb_pca, axis=1, keepdims=True)
norms[norms == 0] = 1e-10 
vectors_norm = emb_pca / norms
fig = components_to_fig_3D_simple(
    components=vectors_norm,
    encoder=encoder,
    base_color=base_colors
)
fig.show()

name_file = f"embedding/norm03/seed_0_SGNS_OneEmbWeighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
norms = np.linalg.norm(emb_pca, axis=1, keepdims=True)
norms[norms == 0] = 1e-10 
vectors_norm = emb_pca / norms
fig = components_to_fig_3D_simple(
    components=vectors_norm,
    encoder=encoder,
    base_color=base_colors
)
fig.show()

NameError: name 'base_colors' is not defined

In [ ]:
x = vectors_norm[:, 0]
y = vectors_norm[:, 1]
z = vectors_norm[:, 2]

# 2. Calculate the radius
r = np.sqrt(x**2 + y**2 + z**2)

azimuth_radians = np.arctan2(y, x)
elevation_radians = np.arcsin(z / r)

idx_to_word = {idx: word for word, idx in encoder.items()}
words_ordered = [idx_to_word.get(i, f"vec_{i}") for i in range(vectors_norm.shape[0])]

colors = [base_colors.get(w, "gray") for w in words_ordered]

fig = plt.figure()
ax = fig.add_subplot(111, projection='mollweide')

ax.scatter(azimuth_radians, elevation_radians)
plt.show()

In [ ]:
fig = components_to_fig_mollweide(    components=emb_pca,
    encoder=encoder,
    base_color=base_colors)
fig.show()

# Other

In [ ]:
name_file = f"embedding/GNG_IWW/norm02/seed_0_SGNS_OneEmbWeighted_.npz"
model = np.load(name_file)
vecs = model['vectors']
words = model['words']

decoder = {index : str(value) for index, value in enumerate(words)}
encoder = {str(value) : index for index, value in enumerate(words)}
    
base_colors = {
    'banana': ("yellow",  "lightyellow"),
    'gorilla': ("gray", "lightgray"),
    'zookeeper': ("brown", "sandybrown"),
    "little": ("pink", "lightpink"),
    "yellow": ("gold", "lightgoldenrodyellow"),
    'light' : ("gold", "lightgoldenrodyellow"),
    'turned': ('limegreen', 'green'),
    'spot': ('beige', 'bisque'),
    'finding': ('red', 'firebrick'),
}

pca = PCA(n_components=3)
emb_pca = pca.fit_transform(vecs)
    
# fig = components_to_fig_3D(
#     components=emb_pca,
#     encoder=encoder,
#     highlight_words=cluster_animal,
#     nb_neighbors=1, base_color=base_colors
# )

# fig.show()

# fig = components_to_fig_3D(
#     components=emb_pca,
#     encoder=encoder,
#     highlight_words=["cat", "cow", "horse", "duck", "pig", "dog"],
#     nb_neighbors=1, base_color=base_colors
# )

# fig.show()

base_colors = {}
cluster_IWW = ["cat", "cow", "horse", "duck", "pig", "dog"]
for w_IWW in cluster_IWW:
    base_colors[w_IWW] = ("green", "lightyellow")
    
for w_gng in cluster_animal:
    base_colors[w_gng] = ("blue", "lightyellow")
print(base_colors)
print(cluster_IWW)
print(cluster_animal)
cluster_IWW.extend(cluster_animal)
print(cluster_IWW)
fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_IWW,
    words_display=None,
    nb_neighbors=1, base_color=base_colors
)

fig.show()

fig = components_to_fig_3D(
    components=emb_pca,
    encoder=encoder,
    highlight_words=cluster_IWW,
    words_display=cluster_IWW,
    nb_neighbors=1, base_color=base_colors
)
fig.show()